# collection-agent — matcher experiments

Turn a **messy DJ list** (`artist`, `title`, with typos) into confident
**Discogs release matches**, by fuzzy-matching against the ETL-published
catalog (`data/published/duckdb/discogs.duckdb`, ~19M releases).

Read-only & offline: no Discogs API, no writes, no imports from `etl/`/`agent/`.

Three experiments below:
1. **Messy-input demo** — run the matcher on a sample batch list.
2. **Accuracy eval** — corrupt *known* releases with typos, measure hit rate.
3. **Disambiguation** — one title, many pressings: what extra signals pick the right one.

In [ ]:
import sys, random
from pathlib import Path
import pandas as pd
pd.set_option('display.max_colwidth', 40)

# import the matcher package from collection-agent/src (parent of notebooks/ is collection-agent/)
_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(_root / 'src'))
from collection_matcher.matcher import Matcher, normalize, split_artist_title

# fast_index=True: ~6s build + ~7 GB RAM, then sub-second queries (good for the eval loop).
# Flip to False if memory-constrained: ~5s per query, no extra RAM.
FAST_INDEX = True
m = Matcher(fast_index=FAST_INDEX)
# DJ sold you records -> constrain matches to vinyl pressings.
VINYL_ONLY = True
print('catalog:', m.db_path)
print('releases:', m.con.execute(f'SELECT count(*) FROM {m._from}').fetchone()[0])

## 1. Messy-input demo

`data/sample_batches.csv` mimics the DJ's list — deliberate typos
(`Kraftwrk`, `Autoban`, `Mezzanin`, `Bladerunner`...). For each row we take the
top candidate and flag it `ambiguous` when the runner-up is within 0.02 of the
top score (i.e. the matcher can't confidently pick one pressing).

In [ ]:
batches = pd.read_csv('../data/sample_batches.csv') if Path('../data/sample_batches.csv').exists() else pd.read_csv('data/sample_batches.csv')

rows = []
for r in batches.itertuples():
    res = m.match_one(r.artist, r.title, k=5, artist_weight=0.4, vinyl_only=VINYL_ONLY)
    b = res.get('best', {})
    rows.append({
        'batch': r.batch,
        'in_artist': r.artist, 'in_title': r.title,
        'match_artist': b.get('artist'), 'match_title': b.get('title'),
        'year': b.get('year'), 'country': b.get('country'), 'fmt': b.get('fmt'),
        'score': res.get('score'), 'ambiguous': res.get('ambiguous'),
        'release_id': res.get('release_id'),
    })
pd.DataFrame(rows)

Note how the typo'd inputs still resolve to the right work, and that almost
everything is flagged `ambiguous` — because popular records have **many
pressings** that tie on artist+title. That's the real problem (experiment 3),
not the fuzzy match itself.

## 2. Accuracy eval (ground truth)

We can't grade matching without known-correct answers. So: sample N real
releases, **corrupt** their artist+title with random typos to simulate the
DJ's list, run the matcher, and measure:

- **work-level top-1** — does the #1 candidate have the right (artist, title)?
- **tied pressings** — how many candidates tie the top score (capped at k)?
- **exact pressing as #1** — is the *original* `release_id` the #1 result?

The first should be high (fuzzy matching the *work* is easy); the last is
essentially luck when many pressings tie — that gap is the disambiguation
problem, quantified.

In [ ]:
rng = random.Random(42)
def corrupt(s):
    s = list(str(s))
    if len(s) > 4:
        i = rng.randrange(len(s) - 1)
        op = rng.choice(['swap', 'drop', 'dup', 'typo'])
        if op == 'swap': s[i], s[i+1] = s[i+1], s[i]
        elif op == 'drop': del s[i]
        elif op == 'dup': s.insert(i, s[i])
        elif op == 'typo': s[i] = rng.choice('abcdefghijklmnopqrstuvwxyz')
    return ''.join(s)

# filter FIRST, then sample (USING SAMPLE applies before WHERE, so wrap it)
truth = m.con.execute(f'''
    SELECT release_id, artist, title, year, country, fmt FROM (
        SELECT release_id, artist, title, year, country, fmt, na, nt
        FROM {m._from}
        WHERE artist <> 'Various' AND nt <> '' AND na <> '' AND year IS NOT NULL
    ) USING SAMPLE 25 ROWS (reservoir, 42)
''').df()

K = 10
ev = []
for t in truth.itertuples():
    ca, ct = corrupt(t.artist), corrupt(t.title)
    cands = m.search(ca, ct, k=K, artist_weight=0.4)
    top = cands.iloc[0]
    work_hit = (normalize(top.artist) == normalize(t.artist)) and (normalize(top.title) == normalize(t.title))
    n_tied = int((cands.score >= top.score - 0.001).sum())
    exact_top1 = int(t.release_id) == int(top.release_id)
    ev.append({'true_artist': t.artist, 'true_title': t.title,
               'corrupted': f'{ca} / {ct}', 'work_top1': work_hit,
               'tied_pressings': n_tied, 'exact_pressing_top1': exact_top1})
ev = pd.DataFrame(ev)
print(f'rows evaluated              : {len(ev)}')
print(f'work-level top-1 hit rate   : {ev.work_top1.mean():.0%}')
print(f'mean tied pressings (cap {K}) : {ev.tied_pressings.mean():.1f}')
print(f'ambiguous (>1 tied) share   : {(ev.tied_pressings > 1).mean():.0%}')
print(f'exact pressing as #1        : {ev.exact_pressing_top1.mean():.0%}  <- luck without more signal')
ev

## 3. Disambiguation: one title, many pressings

`jaro_winkler` ties dozens of pressings of the same record. To pick the one you
physically own you need signals the DJ's list doesn't have — but that you can
surface for a human (or filter on if you know format/country). The strongest is
the label **catalog number** (`catno`), which lives in `release_label_bridge`.

In [ ]:
cands = m.search('Kraftwerk', 'Autobahn', k=8, artist_weight=0.4, vinyl_only=VINYL_ONLY)
ids = [int(x) for x in cands.release_id]
# join catno from the label bridge (same published DuckDB, mode-agnostic name)
catno = m.con.execute(f'''
    SELECT release_id, any_value(catno) AS catno
    FROM {m.src_table('release_label_bridge')} WHERE release_id IN ({','.join(map(str, ids))})
    GROUP BY release_id
''').df() if not cands.empty else pd.DataFrame()
out = cands.merge(catno, on='release_id', how='left') if not catno.empty else cands
print(f'{len(cands)} candidates tie around score {cands.score.max()} — disambiguate by year/country/fmt/catno:')
out[['release_id','title','artist','year','country','fmt','label','catno','score']]

## 4. Your real batch (`data/pending_discogs.csv`)

Optional, gitignored. Your DJ list is one `title` column shaped `ARTIST - Title`
(plus a `batch` column), so we `split_artist_title()` first, then match. Results
are sorted **worst-score first** — that's your review queue; the high-confidence
tail is ready to import once the write side exists. Drop your own file there to
use it; the cell no-ops if it's absent.

In [ ]:
real_path = next((q for q in [Path('../data/pending_discogs.csv'), Path('data/pending_discogs.csv')] if q.exists()), None)
if real_path is None:
    print('No pending_discogs.csv found — drop a CSV with columns [batch, title] (title = "Artist - Title").')
else:
    real = pd.read_csv(real_path)
    out = []
    for r in real.itertuples():
        artist, title = split_artist_title(r.title)
        res = m.match_one(artist, title, k=5, artist_weight=0.35, vinyl_only=VINYL_ONLY)
        b = res.get('best', {})
        out.append({'batch': r.batch, 'raw': r.title, 'artist': artist, 'title': title,
                    'match_artist': b.get('artist'), 'match_title': b.get('title'),
                    'year': b.get('year'), 'country': b.get('country'), 'fmt': b.get('fmt'),
                    'release_id': res.get('release_id'), 'score': res.get('score'),
                    'ambiguous': res.get('ambiguous')})
    real_out = pd.DataFrame(out)
    confident = (real_out.score >= 0.95) & (~real_out.ambiguous)
    print(f'{len(real_out)} rows across {real_out.batch.nunique()} batches')
    print(f'high-confidence (score>=0.95, unambiguous): {confident.sum()}')
    print(f'needs review (low score or tied pressings) : {(~confident).sum()}')
    real_out.sort_values('score').reset_index(drop=True).head(20)  # worst first = review queue

### Takeaways

- **Fuzzy matching the *work* is basically solved** with a normalized
  Jaro-Winkler scan — typos don't sink it, and it runs over the full 19M-row
  dump in seconds.
- **Picking the *pressing* is the hard part**: many pressings tie on score, so
  the exact-pressing-as-#1 rate is low — `artist+title` alone can't choose.
- **Next:** fold `year`/`country`/`fmt`/`catno` into scoring or a review step,
  then build the Discogs API write side (add-to-collection + per-batch folders).
  Both are net-new; see `../README.md`.